# Docker for Data Science Projects: A Beginner-Friendly Introduction

Run in a disposable Linux guest with a real rootless Docker engine, no shared host folders and no public service forwarding. Requires Python, ipykernel, nbclient, nbformat, pexpect and htpasswd (Ubuntu apache2-utils). Commands create/delete only uniquely named and labelled fixture resources after verifying IDs. Registry credentials are temporary and never printed; registry binds only guest loopback. No Docker Hub account, public image upload, paid API or GPU is used.

The complete application files also live in `../../apps/docker-introduction/`. Source command synopses/placeholders remain in the appendix; maintained examples use actual owned resources.

In [ ]:
import os, sys, json, uuid, tempfile, subprocess, shutil, platform, hashlib, time
from pathlib import Path
if platform.system() != 'Linux' or not shutil.which('docker'):
    raise RuntimeError('Use a disposable Linux VM with a real running Docker engine.')
endpoint = subprocess.check_output(['docker', 'context', 'inspect', '--format', '{{.Endpoints.docker.Host}}'], text=True).strip()
assert endpoint.startswith('unix://'), 'Only a local engine socket is permitted'
temporary = tempfile.TemporaryDirectory(prefix='tdb-docker-')
work = Path(temporary.name)
config_dir = work / 'docker-config'; config_dir.mkdir()
docker_env = dict(os.environ, DOCKER_HOST=endpoint, DOCKER_CONFIG=str(config_dir))
run_id = uuid.uuid4().hex[:12]; prefix = 'tdb26-' + run_id
owner_label = 'tdb.archive26.run=' + run_id
containers = {}; image_refs = {}; volumes = set(); build_logs = {}
def inspect_container(name):
    p = subprocess.run(['docker', 'inspect', name], env=docker_env, text=True, capture_output=True)
    return json.loads(p.stdout)[0] if p.returncode == 0 else None
def cli(args, *, quiet=False, check=True, input=None, timeout=1200):
    if args[0] in ['stop', 'rm']:
        for name in [x for x in args[1:] if not x.startswith('-')]:
            value = inspect_container(name)
            assert value and value['Config']['Labels'].get('tdb.archive26.run') == run_id
            assert containers.get(name) == value['Id'], 'Container identity changed'
    if args[0] == 'rmi':
        for name in [x for x in args[1:] if not x.startswith('-')]:
            assert name in image_refs
            actual = subprocess.check_output(['docker','image','inspect','--format','{{.Id}}',name],env=docker_env,text=True).strip()
            assert image_refs[name] == actual, 'Image reference changed'
    p = subprocess.run(['docker', *args], cwd=work, env=docker_env, text=True, input=input, capture_output=True, timeout=timeout)
    if check and p.returncode: raise RuntimeError(p.stderr)
    if not quiet:
        print(p.stdout, end='')
        if p.stderr: print('Docker stderr:', p.stderr.strip())
    return p
def image_info(ref): return json.loads(cli(['image','inspect',ref],quiet=True).stdout)[0]
def remember_image(ref):
    image_refs[ref] = image_info(ref)['Id']
    return ref
def create_container(suffix, *, running=True, image='ubuntu:26.04'):
    name = prefix + '-' + suffix
    assert inspect_container(name) is None
    args = ['run','-d'] if running else ['create']
    cli([*args,'--name',name,'--label',owner_label,image,'sleep','1200'],quiet=True)
    containers[name] = inspect_container(name)['Id']
    return name
def tag(source, target):
    assert target.startswith(prefix + '/') or target.startswith('localhost:')
    assert cli(['image','inspect',target],quiet=True,check=False).returncode != 0
    cli(['tag',source,target],quiet=True); remember_image(target)
    return target
def cleanup():
    for name in list(containers):
        if inspect_container(name): cli(['rm','-f','-v',name],quiet=True)
    for name in list(volumes):
        info = json.loads(cli(['volume','inspect',name],quiet=True).stdout)[0]
        assert info['Labels'].get('tdb.archive26.run') == run_id
        cli(['volume','rm',name],quiet=True)
    for ref in list(image_refs):
        if cli(['image','inspect',ref],quiet=True,check=False).returncode == 0:
            cli(['rmi',ref],quiet=True)
    temporary.cleanup()
engine = json.loads(cli(['info','--format','{{json .}}'],quiet=True).stdout)
assert engine['OSType'] == 'linux'
print('Python:', sys.version.split()[0], 'Actual Docker engine:', engine['ServerVersion'])
print('Linux kernel:', platform.release(), 'Architecture:', platform.machine())
print('Only uniquely labelled fixture containers, image references and volumes may be removed.')
assert any('rootless' in value for value in engine['SecurityOptions'])
command_logs = []
print('Rootless local engine confirmed; no public registry credentials are used.')

In [ ]:
cli(['pull', 'ubuntu:26.04'], quiet=True)
print('Ubuntu resolved digest:', image_info('ubuntu:26.04')['RepoDigests'][0])
listing = cli(['run', '--rm', '--label', owner_label, 'ubuntu:26.04', 'ls', '/'], quiet=True).stdout
assert 'etc' in listing.split() and 'usr' in listing.split()
print(listing, end='')
running = create_container('running')
print('Owned running container:', running)

In [ ]:
visible = cli(['ps', '--filter', 'label=' + owner_label, '--format', '{{.ID}} {{.Names}} {{.Status}}'], quiet=True).stdout
print(visible, end='')
assert running in visible
all_owned = cli(['ps', '-a', '--filter', 'label=' + owner_label, '--format', '{{.Names}}'], quiet=True).stdout
assert running in all_owned.splitlines()
print('docker ps -a lists this run’s containers:', all_owned.strip())

In [ ]:
cli(['stop', running])
assert inspect_container(running)['State']['Running'] is False
assert running not in cli(['ps', '--format', '{{.Names}}'], quiet=True).stdout.splitlines()
assert running in cli(['ps', '-a', '--format', '{{.Names}}'], quiet=True).stdout.splitlines()
print('Stopped, but still present until removal.')

In [ ]:
cli(['rm', running])
assert inspect_container(running) is None
stopped = [create_container('stopped-' + str(i), running=False) for i in range(3)]
cli(['rm', *stopped])
assert all(inspect_container(name) is None for name in stopped)
print('Removed exactly four owned containers; labels and captured IDs were checked.')

In [ ]:
rows = cli(['images', 'ubuntu:26.04', '--format', '{{.Repository}}:{{.Tag}} {{.ID}}'], quiet=True).stdout
assert 'ubuntu:26.04' in rows
print(rows, end='')

In [ ]:
aliases = [tag('ubuntu:26.04', prefix + '/example-' + str(i) + ':latest') for i in range(3)]
cli(['rmi', aliases[0]])
cli(['rmi', *aliases[1:]])
assert all(cli(['image', 'inspect', ref], quiet=True, check=False).returncode != 0 for ref in aliases)
assert image_info('ubuntu:26.04')['Id']
print('Only the three owned image references were removed; the shared Ubuntu image remains.')

In [ ]:
context = work / 'command-context'; context.mkdir()
(context / 'marker.txt').write_text('Docker build fixture\n')
(context / 'Dockerfile').write_text('FROM ubuntu:26.04\nCOPY marker.txt /opt/fixture.txt\nCMD ["cat", "/opt/fixture.txt"]\n')
built_ref = prefix + '/command-example:latest'
build = cli(['build', '--progress=plain', '--label', owner_label, '-t', built_ref, str(context)], quiet=True)
build_logs['command-example'] = build.stdout + build.stderr
remember_image(built_ref)
result = cli(['run', '--rm', '--label', owner_label, built_ref], quiet=True).stdout
assert result == 'Docker build fixture\n'
print('Built and ran:', built_ref)
print(result, end='')

In [ ]:
exec_container = create_container('exec')
listing = cli(['exec', exec_container, 'ls', '/'], quiet=True).stdout
assert 'etc' in listing.split()
print(listing, end='')
import pexpect, re
child = pexpect.spawn('docker', ['exec', '-it', exec_container, 'bash'], env=docker_env, encoding='utf-8', timeout=20)
child.sendline('printf "PTY_RESULT=%s\\n" "$((3+4))"; exit')
child.expect(pexpect.EOF)
transcript = re.sub(r'\x1b\[[0-9;?]*[A-Za-z]', '', child.before).replace('\r', '')
child.close()
assert child.exitstatus == 0 and re.search(r'^PTY_RESULT=7$', transcript, re.M)
print('Actual interactive docker exec -it shell computed PTY_RESULT=7.')

In [ ]:
cli(['pull', 'ubuntu:26.04'], quiet=True)
print('Explicit supported tag:', image_info('ubuntu:26.04')['RepoDigests'][0])
cli(['pull', 'python:3.13-slim'], quiet=True)
print('Official Python tag:', image_info('python:3.13-slim')['RepoDigests'][0])

In [ ]:
import secrets, socket, urllib.request, base64
cli(['pull','registry:3'],quiet=True)
with socket.socket() as probe:
    probe.bind(('127.0.0.1',0));registry_port=probe.getsockname()[1]
auth_dir=work/'registry-auth';auth_dir.mkdir()
test_password=secrets.token_urlsafe(24)
hashed=subprocess.check_output(['htpasswd','-Bbn','tutorial',test_password],text=True)
(auth_dir/'htpasswd').write_text(hashed)
registry_name=prefix+'-registry'
cli(['run','-d','--name',registry_name,'--label',owner_label,'-p',f'127.0.0.1:{registry_port}:5000','--mount',f'type=bind,source={auth_dir},target=/auth,readonly','-e','REGISTRY_AUTH=htpasswd','-e','REGISTRY_AUTH_HTPASSWD_REALM=Tutorial','-e','REGISTRY_AUTH_HTPASSWD_PATH=/auth/htpasswd','registry:3'],quiet=True)
containers[registry_name]=inspect_container(registry_name)['Id']
registry_address=f'localhost:{registry_port}'
for attempt in range(60):
    p=cli(['login',registry_address,'--username','tutorial','--password-stdin'],quiet=True,check=False,input=test_password+'\n')
    if p.returncode==0:break
    time.sleep(0.5)
else:raise RuntimeError(p.stderr)
wrong=cli(['login',registry_address,'--username','tutorial','--password-stdin'],quiet=True,check=False,input='incorrect-test-password\n')
assert wrong.returncode != 0
cli(['login',registry_address,'--username','tutorial','--password-stdin'],quiet=True,input=test_password+'\n')
local_ref=tag(built_ref,registry_address+'/'+prefix+'/my-image:1.0')
cli(['push',local_ref],quiet=True)
first_digest=image_info(local_ref)['RepoDigests']
assert first_digest
cli(['rmi',local_ref],quiet=True)
cli(['pull',local_ref],quiet=True);remember_image(local_ref)
assert cli(['run','--rm','--label',owner_label,local_ref],quiet=True).stdout=='Docker build fixture\n'
print('Actual authenticated push/pull round-trip to a loopback-only registry passed; wrong credentials were rejected.')
print('Registry manifest digest:',first_digest[0].rsplit('@',1)[-1])

In [ ]:
files = {'iris_classification.py': "import json\nimport numpy as np\nfrom sklearn.datasets import load_iris\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import accuracy_score, confusion_matrix\n\ndef train_and_evaluate():\n    iris = load_iris()\n    X_train, X_test, y_train, y_test = train_test_split(\n        iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target)\n    model = LogisticRegression(max_iter=300, random_state=42).fit(X_train, y_train)\n    prediction = model.predict(X_test)\n    assert np.isfinite(model.coef_).all()\n    return {'dataset': 'iris', 'train_rows': len(y_train), 'test_rows': len(y_test),\n            'accuracy': float(accuracy_score(y_test, prediction)),\n            'correct': int((prediction == y_test).sum()),\n            'predictions': prediction.tolist(), 'targets': y_test.tolist(),\n            'confusion_matrix': confusion_matrix(y_test, prediction).tolist()}\n\nif __name__ == '__main__':\n    print(json.dumps(train_and_evaluate(), sort_keys=True))\n", 'requirements.txt': 'scikit-learn==1.9.1\nnumpy==2.4.6\nscipy==1.17.1\njoblib==1.6.0\nthreadpoolctl==3.7.0\n', 'Dockerfile': 'FROM python:3.13-slim@sha256:2d44738ebf4ce5f63b9d9ea871e9c7cfdc0426f7b0c97cac915fd88ded1d86bb\nWORKDIR /app\nCOPY requirements.txt ./\nRUN python -m pip install --no-cache-dir -r requirements.txt\nRUN useradd --create-home --uid 10001 --user-group app\nCOPY --chown=app:app iris_classification.py ./\nUSER app\nCMD ["python", "iris_classification.py"]\n', '.dockerignore': '*\n!Dockerfile\n!requirements.txt\n!iris_classification.py\n'}
app_context = work / 'iris-app'; app_context.mkdir()
for name, text in files.items():
    (app_context / name).write_text(text)
assert len(files) == 4
print('Complete build inputs:', ', '.join(sorted(files)))

In [ ]:
print(files['iris_classification.py'])

In [ ]:
print(files['requirements.txt'])
print('Use a clean project environment; freeze records installed versions but does not prove dependency compatibility.')

In [ ]:
print(files['Dockerfile'])

In [ ]:
app_ref = prefix + '/iris-classification:test'
build = cli(['build', '--progress=plain', '--label', owner_label, '-t', app_ref, str(app_context)], quiet=True)
build_logs['iris-app'] = build.stdout + build.stderr
remember_image(app_ref)
actual = json.loads(cli(['run', '--rm', '--label', owner_label, app_ref], quiet=True).stdout)
print(json.dumps(actual, indent=2, sort_keys=True))

In [ ]:
assert actual['train_rows'] == 120 and actual['test_rows'] == 30
predictions, targets = actual['predictions'], actual['targets']
assert len(predictions) == len(targets) == 30
correct = sum(p == t for p, t in zip(predictions, targets))
assert actual['correct'] == correct and actual['accuracy'] == correct / 30
matrix = [[sum(t == row and p == col for p, t in zip(predictions, targets)) for col in range(3)] for row in range(3)]
assert actual['confusion_matrix'] == matrix
assert sum(sum(row) for row in matrix) == 30
again = json.loads(cli(['run', '--rm', '--label', owner_label, app_ref], quiet=True).stdout)
assert again == actual
uid = cli(['run', '--rm', '--label', owner_label, app_ref, 'id', '-u'], quiet=True).stdout.strip()
assert uid == '10001'
remote_code = cli(['run', '--rm', '--label', owner_label, app_ref, 'python', '-c',
    "import hashlib;print(hashlib.sha256(open('/app/iris_classification.py','rb').read()).hexdigest())"], quiet=True).stdout.strip()
assert remote_code == hashlib.sha256(files['iris_classification.py'].encode()).hexdigest()
packages = cli(['run', '--rm', '--label', owner_label, app_ref, 'python', '-m', 'pip', 'freeze'], quiet=True).stdout
assert all(line in packages.splitlines() for line in files['requirements.txt'].splitlines())
print('Non-root UID:', uid)
print('PASS: independent held-out accuracy/confusion matrix, deterministic repeat, exact application hash and installed pinned versions.')

In [ ]:
assert all(inspect_container(name) is None or inspect_container(name)['Config']['Labels'].get('tdb.archive26.run') == run_id for name in containers)
assert image_info(local_ref)['Id'] == image_info(built_ref)['Id']
print('PASS: all ten real Docker command families, interactive PTY, authenticated loopback-only registry and complete ML application.')
cleanup()
assert all(inspect_container(name) is None for name in containers)
print('Removed only this run’s labelled containers, image aliases, test registry and temporary credentials; shared caches remain.')

## Original source-code appendix

All 28 source blocks are retained verbatim as nonexecuting historical source, including placeholder command synopses and the old Dockerfile.

```text
docker run [OPTIONS] IMAGE [COMMAND] [ARG...]
```

```text
docker run ubuntu ls
```

```text
docker ps [OPTIONS]
```

```text
docker ps [OPTIONS]
```

```text
docker ps -a
```

```text
docker stop [OPTIONS] CONTAINER [CONTAINER...]
```

```text
docker stop my-container
```

```text
docker rm [OPTIONS] CONTAINER [CONTAINER...]
```

```text
docker rm my-container
```

```text
docker rm container1 container2 container3
```

```text
docker images [OPTIONS] [REPOSITORY[:TAG]]
```

```text
docker images
```

```text
docker rmi [OPTIONS] IMAGE [IMAGE...]
```

```text
docker rmi my-image:latest
```

```text
docker rmi image1 image2 image3
```

```text
docker build [OPTIONS] PATH | URL | -
```

```text
docker build -t my-image:latest .
```

```text
docker exec [OPTIONS] CONTAINER COMMAND [ARG...]
```

```text
docker exec my-container ls
```

```text
docker exec -it my-container bash
```

```text
docker pull [OPTIONS] IMAGE[:TAG]
```

```text
docker pull ubuntu
```

```text
docker pull ubuntu:20.04
```

```text
docker push [OPTIONS] NAME[:TAG]
```

```text
docker push username/my-image:latest
```

```text
# Load the libraries

from sklearn.datasets import load_iris

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score


# Load the iris dataset

iris = load_iris()

X = iris.data

y = iris.target


# Split the data

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)


# Train a logistic regression model

clf = LogisticRegression()

clf.fit(X_train, y_train)


# Make predictions

y_pred = clf.predict(X_test)


# Print the accuracy of the model

accuracy_score = accuracy_score(y_test, y_pred)

print(f'Accuracy: {accuracy_score}')
```

```text
pip3 freeze > requirements.txt  # Python3
```

```text
FROM python:3.9

WORKDIR /src

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY . .

CMD ["python","iris_classification.py"]
```